In [ ]:
from datetime import timedelta
from io import BytesIO

import polars as pl
import requests

In [ ]:
url = "https://services.swpc.noaa.gov/json/goes/primary/xrays-7-day.json"
r = requests.get(url)
r.raise_for_status()

In [ ]:
energy = pl.Enum(["0.05-0.4nm", "0.1-0.8nm"])

In [ ]:
df = pl.read_json(
    BytesIO(r.content), schema_overrides={"energy": energy, "flux": pl.Float64}
)
df

In [ ]:
df = df.with_columns(
    pl.col("time_tag")
    .str.to_datetime("%Y-%m-%dT%H:%M:%SZ", time_unit="ns")
    .alias("time")
)

In [ ]:
df

In [ ]:
df = df.filter(pl.col("energy") == "0.1-0.8nm")
df

In [ ]:
df.with_columns(
    pl.col("time").diff().fill_null(pl.duration(minutes=1)).alias("gap")
).filter(pl.col("gap") > pl.duration(minutes=1))

In [ ]:
df.filter(pl.col("time") <= pl.col("time").shift(1))

In [ ]:
df.null_count()

In [ ]:
df = df.select(
    pl.col("time"),
    (pl.col("flux") * 1000000)
    .fill_null(strategy="forward", limit=10)
    .alias("xrsb_flux"),
)
df

In [ ]:
df.filter(pl.col("time") == pl.col("time").max())

In [ ]:
off = pl.scan_parquet("gs://mlops-solar-flux_offline_fs/")

In [ ]:
latest_off_time = off.select(pl.col("time").max()).collect().item()
latest_off_time

In [ ]:
df = df.filter(pl.col("time") > latest_off_time)
df

In [ ]:
if df.height == 0:
    print("no new features to calculate")

In [ ]:
earliest_op_time = df.select(pl.col("time").min()).item()
earliest_op_time

In [ ]:
cutoff_time = earliest_op_time - timedelta(minutes=1440)
cutoff_time

In [ ]:
hist = (
    off.filter((pl.col("time") >= cutoff_time) & (pl.col("time") <= latest_off_time))
    .select(pl.col("time"), pl.col("xrsb_flux"))
    .collect()
)
hist

In [ ]:
hist.upsample("time", every="1m")

In [ ]:
merge = pl.concat([hist, df])
merge

In [ ]:
merge.with_columns(
    pl.col("time").diff().fill_null(pl.duration(minutes=1)).alias("gap")
).filter(pl.col("gap") > pl.duration(minutes=1))

In [ ]:
merge.filter(pl.col("time") <= pl.col("time").shift(1))

In [ ]:
merge = merge.upsample("time", every="1m")
merge

In [ ]:
merge.with_columns(
    pl.col("time").diff().fill_null(pl.duration(minutes=1)).alias("gap")
).filter(pl.col("gap") > pl.duration(minutes=1))

In [ ]:
WINDOW_24H = 1440
WINDOW_12H = 720
MIN_FRAC = 0.8
MIN_24H = int(WINDOW_24H * MIN_FRAC)
MIN_12H = int(WINDOW_12H * MIN_FRAC)
C_CLASS_THRESHOLD = 1e-6 * 1000000

In [ ]:
df_feat = (
    merge.lazy()
    .with_columns(
        # Target
        pl.col("xrsb_flux")
        .rolling_max(WINDOW_24H, min_samples=MIN_24H)
        .shift(-WINDOW_24H)
        .alias("max_24h_la"),
        # Lag features
        pl.col("xrsb_flux").shift(15).alias("lag_15"),
        pl.col("xrsb_flux").shift(60).alias("lag_60"),
        pl.col("xrsb_flux").shift(120).alias("lag_120"),
        pl.col("xrsb_flux").shift(1440).alias("lag_1440"),
        # Rolling features
        pl.col("xrsb_flux")
        .rolling_max(WINDOW_12H, min_samples=MIN_12H)
        .alias("roll_max_720"),
        pl.col("xrsb_flux")
        .rolling_std(WINDOW_12H, min_samples=MIN_12H)
        .alias("roll_std_720"),
        pl.col("xrsb_flux")
        .rolling_mean(WINDOW_12H, min_samples=MIN_12H)
        .alias("roll_mean_720"),
        pl.col("xrsb_flux").diff(5).alias("deriv_1_5"),
        pl.col("xrsb_flux").diff(n=5).diff(n=5).alias("deriv_2_5"),
        (
            (pl.col("xrsb_flux") >= C_CLASS_THRESHOLD)
            & (pl.col("xrsb_flux").shift(1) < C_CLASS_THRESHOLD)
        )
        .rolling_sum(WINDOW_12H, min_samples=MIN_12H)
        .alias("roll_c_class_cross_720"),
    )
    .collect()
)

In [ ]:
df_feat

In [ ]:
df_feat.filter(pl.col("time") >= earliest_op_time)

In [ ]:
df_feat.drop_nulls()

In [ ]:
df_feat.filter(pl.col("time") == pl.col("time").max())